# Personal Expense Analyzer

## 1. Project objective
This notebook loads a raw CSV of personal expenses (in Ethiopian birr), cleans it using functions, and processes it with NumPy.

It answers three questions: what is the **total expenditure**, what is the **average expense**, and **which month had the highest spending**.

**Steps:** load -> inspect -> clean -> NumPy analysis -> results -> tests -> conclusion

## 2. Imports

In [1]:
import numpy as np
import pandas as pd

## 3. Load the dataset
The CSV file must be in the same folder as this notebook.
If you run this in Google Colab, the cell below will ask you to upload it.

In [2]:
file_path = "personal_expenses_self_created.csv"


def load_data(file_path):
    """Read the CSV file and return a table (DataFrame)."""
    try:
        data = pd.read_csv(file_path)
        return data
    except FileNotFoundError:
        print("Error: file not found ->", file_path)
        return None


raw = load_data(file_path)

if raw is None:
    try:
        from google.colab import files
        files.upload()
        raw = load_data(file_path)
    except ImportError:
        print("Put the CSV in the same folder as this notebook.")

## 4. Inspect the raw data

In [3]:
print("Size (rows, columns):", raw.shape)
print()
print(raw.head(10))

Size (rows, columns): (50, 5)

         date          category amount_etb payment_method  \
0  2026-09-01  Food & Groceries      850.0           Cash   
1  2026-09-02    Transportation      120.0   Mobile Money   
2  2026-09-03         Utilities      650.0  Bank Transfer   
3  2026-09-04   Dining & Coffee      240.0   Mobile Money   
4  2026-09-05            Health      180.0           Cash   
5  2026-09-06     Personal Care      320.0   Mobile Money   
6  2026-09-07    entertainment       150.0           Cash   
7  2026-09-08    Transportation       90.0   Mobile Money   
8  2026-09-09  Food & Groceries      460.0  Bank Transfer   
9  2026-09-10         Education      500.0   Mobile Money   

              description  
0        Weekly groceries  
1               Taxi fare  
2        Electricity bill  
3     Lunch with a friend  
4       Pharmacy purchase  
5  Haircut and toiletries  
6            Movie ticket  
7                Bus fare  
8    Fruit and vegetables  
9       Online co

In [4]:
print("Column types:")
print(raw.dtypes)
print()
print("Missing values per column:")
print(raw.isna().sum())
print()
print("Duplicate rows:", raw.duplicated().sum())

Column types:
date              str
category          str
amount_etb        str
payment_method    str
description       str
dtype: object

Missing values per column:
date              0
category          1
amount_etb        0
payment_method    0
description       0
dtype: int64

Duplicate rows: 1


**What I notice:**
- `amount_etb` is stored as text (object), so some value is not a number.
- One row has no category.
- One category has extra spaces and small letters (` entertainment `).
- One row is repeated.

In [5]:
bad_amounts = raw[pd.to_numeric(raw["amount_etb"], errors="coerce").isna()]
print(bad_amounts)

          date   category    amount_etb payment_method        description
25  2026-09-26  Education  not recorded           Cash  Notebook and pens


## 5. Cleaning functions
Each function does only one job.

In [6]:
def remove_duplicates(data):
    """Remove rows that are exactly the same as another row."""
    data = data.drop_duplicates()
    return data


def clean_category_text(data):
    """Remove extra spaces and make the capital letters consistent."""
    data["category"] = data["category"].str.strip().str.title()
    return data


def fill_missing_category(data):
    """Rows with no category get the label 'Uncategorized'."""
    data["category"] = data["category"].fillna("Uncategorized")
    return data


def fix_amount_column(data):
    """Turn the amount column into numbers. Text like 'not recorded' becomes NaN."""
    data["amount_etb"] = pd.to_numeric(data["amount_etb"], errors="coerce")
    return data


def remove_invalid_amounts(data):
    """Drop rows where the amount is missing or not above zero."""
    data = data[data["amount_etb"].notna()]
    data = data[data["amount_etb"] > 0]
    return data


def fix_date_column(data):
    """Convert the date column from text into real dates."""
    data["date"] = pd.to_datetime(data["date"], errors="coerce")
    data = data[data["date"].notna()]
    return data


def clean_data(data):
    """Run all cleaning steps in order and return the cleaned table."""
    data = data.copy()
    data = remove_duplicates(data)
    data = clean_category_text(data)
    data = fill_missing_category(data)
    data = fix_amount_column(data)
    data = remove_invalid_amounts(data)
    data = fix_date_column(data)
    data = data.reset_index(drop=True)
    return data

## 6. Run the cleaning and compare raw vs clean

In [7]:
clean = clean_data(raw)

print("Raw size:  ", raw.shape)
print("Clean size:", clean.shape)

Raw size:   (50, 5)
Clean size: (48, 5)


In [8]:
print("Raw categories:")
print(sorted(raw["category"].dropna().unique()))
print()
print("Clean categories:")
print(sorted(clean["category"].unique()))
print()
print("Clean column types:")
print(clean.dtypes)

Raw categories:
[' entertainment ', 'Dining & Coffee', 'Education', 'Entertainment', 'Family & Gifts', 'Food & Groceries', 'Health', 'Housing', 'Personal Care', 'Savings', 'Shopping', 'Transportation', 'Utilities']

Clean categories:
['Dining & Coffee', 'Education', 'Entertainment', 'Family & Gifts', 'Food & Groceries', 'Health', 'Housing', 'Personal Care', 'Savings', 'Shopping', 'Transportation', 'Uncategorized', 'Utilities']

Clean column types:
date              datetime64[us]
category                     str
amount_etb               float64
payment_method               str
description                  str
dtype: object


## 7. Processing with NumPy
The amounts are converted into a NumPy array, and all the calculations use NumPy functions.

In [9]:
def analyze_amounts(data):
    """Use NumPy to calculate the main statistics on the amount column."""
    amounts = data["amount_etb"].to_numpy()

    results = {
        "count": len(amounts),
        "total": np.sum(amounts),
        "mean": np.mean(amounts),
        "median": np.median(amounts),
        "minimum": np.min(amounts),
        "maximum": np.max(amounts),
    }
    return results


def total_per_category(data):
    """Use NumPy to add up the spending of each category."""
    amounts = data["amount_etb"].to_numpy()
    categories = data["category"].to_numpy()

    totals = {}
    for name in np.unique(categories):
        mask = categories == name
        totals[name] = np.sum(amounts[mask])
    return totals


def total_per_month(data):
    """Use NumPy to add up the spending of each month."""
    amounts = data["amount_etb"].to_numpy()
    months = data["date"].dt.strftime("%Y-%m").to_numpy()

    totals = {}
    for month in np.unique(months):
        mask = months == month
        totals[month] = np.sum(amounts[mask])
    return totals


def find_highest_month(month_totals):
    """Return the month with the highest total spending and its amount."""
    names = list(month_totals.keys())
    values = np.array(list(month_totals.values()))
    position = np.argmax(values)
    return names[position], values[position]

## 8. Results

### Total expenditure and average expense

In [10]:
results = analyze_amounts(clean)

print("Number of transactions:", results["count"])
print("Total expenditure:     ", round(results["total"], 2), "ETB")
print("Average expense:       ", round(results["mean"], 2), "ETB")
print("Median expense:        ", round(results["median"], 2), "ETB")
print("Smallest expense:      ", round(results["minimum"], 2), "ETB")
print("Largest expense:       ", round(results["maximum"], 2), "ETB")

Number of transactions: 48
Total expenditure:      26420.0 ETB
Average expense:        550.42 ETB
Median expense:         305.0 ETB
Smallest expense:       90.0 ETB
Largest expense:        4200.0 ETB


### Which month had the highest spending?

In [11]:
month_totals = total_per_month(clean)

for month, total in month_totals.items():
    print(f"{month}   {total:10.2f} ETB")

top_month, top_total = find_highest_month(month_totals)
print()
print("Month with the highest spending:", top_month, "->", round(top_total, 2), "ETB")

2026-09     15555.00 ETB
2026-10     10865.00 ETB

Month with the highest spending: 2026-09 -> 15555.0 ETB


### Spending per category

In [12]:
category_totals = total_per_category(clean)

sorted_totals = sorted(category_totals.items(), key=lambda item: item[1], reverse=True)
for name, total in sorted_totals:
    print(f"{name:20s} {total:10.2f}")

Housing                 8400.00
Food & Groceries        5810.00
Shopping                2360.00
Utilities               2320.00
Dining & Coffee         1625.00
Transportation          1130.00
Savings                 1000.00
Personal Care            870.00
Education                850.00
Health                   740.00
Family & Gifts           700.00
Entertainment            340.00
Uncategorized            275.00


In [13]:
clean.to_csv("personal_expenses_clean.csv", index=False)
print("Saved personal_expenses_clean.csv")

Saved personal_expenses_clean.csv


## 9. Testing
Checking that the app does not crash on bad input.

In [14]:
print("Test 1: wrong path")
print(load_data("this_file_does_not_exist.csv"))

Test 1: wrong path
Error: file not found -> this_file_does_not_exist.csv
None


In [15]:
print("Test 2: a small table with every problem at once")

test_data = pd.DataFrame({
    "date": ["2026-01-01", "2026-01-01", "2026-01-02", "2026-01-03", "2026-01-04"],
    "category": ["Food", "Food", "  transport ", None, "Food"],
    "amount_etb": ["100", "100", "50", "abc", "-20"],
    "payment_method": ["Cash"] * 5,
    "description": ["a", "a", "b", "c", "d"],
})

test_clean = clean_data(test_data)
print()
print(test_clean)

Test 2: a small table with every problem at once

        date   category  amount_etb payment_method description
0 2026-01-01       Food       100.0           Cash           a
1 2026-01-02  Transport        50.0           Cash           b


In [16]:
print("Test 3: check the results with asserts")

assert len(test_clean) == 2
assert test_clean["amount_etb"].sum() == 150
assert "Transport" in test_clean["category"].values
assert test_clean["category"].isna().sum() == 0
print("All tests passed")

Test 3: check the results with asserts
All tests passed


In [17]:
print("Test 4: month totals on the test data")

test_months = total_per_month(test_clean)
print(test_months)
print(find_highest_month(test_months))

Test 4: month totals on the test data
{'2026-01': np.float64(150.0)}
('2026-01', np.float64(150.0))


In [18]:
print("Test 5: empty table")

empty = pd.DataFrame(columns=["date", "category", "amount_etb", "payment_method", "description"])
if empty.empty:
    print("The table is empty, nothing to process.")

Test 5: empty table
The table is empty, nothing to process.


## 10. Conclusion
- The raw file had **50 rows**. After cleaning, **48 rows** remained (1 duplicate and 1 invalid amount removed, 1 missing category filled, category text fixed).
- **Total expenditure:** 26,420 ETB.
- **Average expense:** about 550.42 ETB per transaction.
- **Highest spending month:** September 2026 with 15,555 ETB, compared to 10,865 ETB in October 2026. (The data for October stops on 19 October, so the month is not complete.)
- The biggest spending category is **Housing**, followed by **Food & Groceries**.

NumPy was used for the sums, the mean, the median, the minimum and maximum, the totals per category and per month, and for finding the highest month.

Without cleaning, the duplicate rent row would have made the total, the Housing category and September wrong, and the text amount would have stopped the calculations.